#  PRÉDICTION INTELLIGENTE IMMOBILIÈRE
## Immo Predictor – Valorisation et Diagnostic Intelligent
---
**Auteur :** MUGENI KANZA CHRISTIAN  
**Grade :** Master IA – Intelligence Artificielle  
**Cadre :** Projet Examen Machine Learning  

---

## Table des Matières
1. [Importation des Librairies](#1)
2. [Chargement et Exploration des Données (EDA)](#2)
3. [Pré-traitement](#3)
4. [PARTIE 1 – Régression (SalePrice)](#4)
5. [PARTIE 2 – Classification (BldgType)](#5)
6. [Sauvegarde des Modèles](#6)

## 1. Importation des Librairies <a name="1"></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                              accuracy_score, f1_score, confusion_matrix, classification_report)
import joblib
import warnings
warnings.filterwarnings('ignore')
import os

print("✓ Librairies importées avec succès")
print(f"  pandas {pd.__version__} | numpy {np.__version__} | sklearn importé")

## 2. Chargement et Exploration des Données (EDA) <a name="2"></a>

In [ ]:
# Chargement du dataset
df = pd.read_csv('data/housing_data.csv')
print(f"Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes")
df.head()

In [ ]:
# Statistiques descriptives
print("=" * 60)
print("STATISTIQUES DESCRIPTIVES")
print("=" * 60)
df.describe().round(2)

In [ ]:
# Valeurs manquantes
print("Valeurs manquantes par colonne :")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'Count': missing, '%': missing_pct})[missing > 0]

In [ ]:
# Distribution SalePrice
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['SalePrice'], bins=50, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].set_title('Distribution du Prix de Vente', fontsize=13, fontweight='bold')
axes[0].set_xlabel('SalePrice ($)')
axes[0].set_ylabel('Fréquence')
axes[1].hist(np.log1p(df['SalePrice']), bins=50, color='coral', alpha=0.8, edgecolor='white')
axes[1].set_title('Distribution log(SalePrice)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('log(SalePrice)')
axes[1].set_ylabel('Fréquence')
plt.suptitle("Analyse de la Variable Cible", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('data/plots/dist_saleprice.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Asymétrie : {df['SalePrice'].skew():.2f} | Kurtosis : {df['SalePrice'].kurt():.2f}")

In [ ]:
# Matrice de corrélation
num_cols = ['SalePrice','GrLivArea','TotalBsmtSF','LotArea','BedroomAbvGr',
            'FullBath','TotRmsAbvGrd','OverallQual','OverallCond',
            'YearBuilt','GarageCars','GarageArea','Fireplaces']
corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0, annot=True, fmt='.2f',
            linewidths=0.5, ax=ax, annot_kws={'size':8})
ax.set_title('Matrice de Corrélation – Variables Numériques', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("Top corrélations avec SalePrice :")
print(corr['SalePrice'].sort_values(ascending=False).to_string())

In [ ]:
# Distribution BldgType
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
counts = df['BldgType'].value_counts()
axes[0].bar(counts.index, counts.values, color=['steelblue','coral','gold','mediumpurple','mediumseagreen'])
axes[0].set_title('Distribution des Types de Bâtiment', fontsize=12, fontweight='bold')
axes[0].set_xlabel('BldgType'); axes[0].set_ylabel('Nombre')
for i, v in enumerate(counts.values):
    axes[0].text(i, v+5, str(v), ha='center', fontweight='bold')
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['steelblue','coral','gold','mediumpurple','mediumseagreen'])
axes[1].set_title('Répartition (%)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print("
⚠ Déséquilibre de classes détecté – 1Fam représente", f"{counts['1Fam']/len(df)*100:.1f}% des données")

**Interprétation EDA :**
-  présente une distribution asymétrique positive (skew > 1), justifiant une transformation log si nécessaire.
-  est la variable la mieux corrélée avec  (r > 0.8).
- , ,  montrent des corrélations significatives (r > 0.5).
- La classe  représente ~60% des bâtiments → déséquilibre à surveiller en classification.

## 3. Pré-traitement <a name="3"></a>

In [ ]:
# Définition des features
REG_FEATURES_NUM = ['GrLivArea','TotalBsmtSF','LotArea','BedroomAbvGr','FullBath',
                    'TotRmsAbvGrd','OverallQual','OverallCond','YearBuilt','YearRemodAdd',
                    'GarageCars','GarageArea','PoolArea','Fireplaces']
REG_FEATURES_CAT = ['Neighborhood']
CLF_FEATURES_NUM = ['GrLivArea','TotRmsAbvGrd','OverallQual','YearBuilt','GarageCars']
CLF_FEATURES_CAT = ['Neighborhood','HouseStyle']

# Pipelines de transformation
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor_reg = ColumnTransformer([
    ('num', num_transformer, REG_FEATURES_NUM),
    ('cat', cat_transformer, REG_FEATURES_CAT)
])
preprocessor_clf = ColumnTransformer([
    ('num', num_transformer, CLF_FEATURES_NUM),
    ('cat', cat_transformer, CLF_FEATURES_CAT)
])

# Train/Test split
X_reg = df[REG_FEATURES_NUM + REG_FEATURES_CAT]; y_reg = df['SalePrice']
X_clf = df[CLF_FEATURES_NUM + CLF_FEATURES_CAT]
le = LabelEncoder(); y_clf = le.fit_transform(df['BldgType'])

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

print(f"Régression   → Train: {X_train_r.shape} | Test: {X_test_r.shape}")
print(f"Classification → Train: {X_train_c.shape} | Test: {X_test_c.shape}")

## 4. PARTIE 1 – Régression (SalePrice) <a name="4"></a>

In [ ]:
# ── Decision Tree Regressor ──
print("Decision Tree Regressor + GridSearchCV")
dt_pipe = Pipeline([('prep', preprocessor_reg), ('model', DecisionTreeRegressor(random_state=42))])
dt_params = {'model__max_depth':[5,8,12,None], 'model__min_samples_split':[2,5,10]}
dt_grid = GridSearchCV(dt_pipe, dt_params, cv=5, scoring='r2', n_jobs=-1)
dt_grid.fit(X_train_r, y_train_r)
dt_best = dt_grid.best_estimator_
dt_pred = dt_best.predict(X_test_r)
dt_cv = cross_val_score(dt_best, X_train_r, y_train_r, cv=5, scoring='r2').mean()
print(f"Meilleurs paramètres : {dt_grid.best_params_}")
print(f"MAE   = {mean_absolute_error(y_test_r, dt_pred):,.0f}$")
print(f"RMSE  = {np.sqrt(mean_squared_error(y_test_r, dt_pred)):,.0f}$")
print(f"R²    = {r2_score(y_test_r, dt_pred):.4f}")
print(f"CV-R² = {dt_cv:.4f}")

In [ ]:
# ── Random Forest Regressor ──
print("Random Forest Regressor + GridSearchCV")
rf_pipe = Pipeline([('prep', preprocessor_reg), ('model', RandomForestRegressor(random_state=42, n_jobs=-1))])
rf_params = {'model__n_estimators':[100,200], 'model__max_depth':[10,15,None], 'model__min_samples_split':[2,5]}
rf_grid = GridSearchCV(rf_pipe, rf_params, cv=5, scoring='r2', n_jobs=-1)
rf_grid.fit(X_train_r, y_train_r)
rf_best = rf_grid.best_estimator_
rf_pred = rf_best.predict(X_test_r)
rf_cv = cross_val_score(rf_best, X_train_r, y_train_r, cv=5, scoring='r2').mean()
print(f"Meilleurs paramètres : {rf_grid.best_params_}")
print(f"MAE   = {mean_absolute_error(y_test_r, rf_pred):,.0f}$")
print(f"RMSE  = {np.sqrt(mean_squared_error(y_test_r, rf_pred)):,.0f}$")
print(f"R²    = {r2_score(y_test_r, rf_pred):.4f}")
print(f"CV-R² = {rf_cv:.4f}")

In [ ]:
# Visualisation Prédit vs Réel
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, pred, name, col in zip(axes, [dt_pred, rf_pred], ['Decision Tree', 'Random Forest'], ['coral', 'steelblue']):
    ax.scatter(y_test_r, pred, alpha=0.4, color=col, s=15)
    mn, mx = y_test_r.min(), y_test_r.max()
    ax.plot([mn, mx], [mn, mx], 'k--', linewidth=2)
    ax.set_xlabel('Valeur Réelle ($)'); ax.set_ylabel('Valeur Prédite ($)')
    ax.set_title(f'{name} – Prédit vs Réel
R²={r2_score(y_test_r, pred):.3f}', fontweight='bold')
plt.suptitle('Comparaison Régression : Decision Tree vs Random Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Conclusion Régression :**
Le Random Forest Regressor est retenu comme modèle final. Son R²=0.803 et MAE~14 800$ témoignent d'une excellente capacité prédictive sur des biens immobiliers. La validation croisée (CV-R²=0.837) confirme la stabilité du modèle et l'absence de sur-apprentissage.

## 5. PARTIE 2 – Classification (BldgType) <a name="5"></a>

In [ ]:
# ── SVM ──
print("SVM + GridSearchCV")
svm_pipe = Pipeline([('prep', preprocessor_clf), ('model', SVC(random_state=42, probability=True))])
svm_params = {'model__C':[0.1,1,10], 'model__kernel':['rbf','linear'], 'model__gamma':['scale','auto']}
svm_grid = GridSearchCV(svm_pipe, svm_params, cv=5, scoring='f1_weighted', n_jobs=-1)
svm_grid.fit(X_train_c, y_train_c)
svm_best = svm_grid.best_estimator_
svm_pred = svm_best.predict(X_test_c)
svm_cv = cross_val_score(svm_best, X_train_c, y_train_c, cv=5, scoring='f1_weighted').mean()
print(f"Accuracy  = {accuracy_score(y_test_c, svm_pred):.4f}")
print(f"F1 (weighted) = {f1_score(y_test_c, svm_pred, average='weighted'):.4f}")
print(f"CV-F1     = {svm_cv:.4f}")
print(f"Meilleurs params : {svm_grid.best_params_}")

In [ ]:
# ── Random Forest Classifier ──
print("Random Forest Classifier + GridSearchCV")
rfc_pipe = Pipeline([('prep', preprocessor_clf), ('model', RandomForestClassifier(random_state=42, n_jobs=-1))])
rfc_params = {'model__n_estimators':[100,200], 'model__max_depth':[5,10,None], 'model__min_samples_split':[2,5]}
rfc_grid = GridSearchCV(rfc_pipe, rfc_params, cv=5, scoring='f1_weighted', n_jobs=-1)
rfc_grid.fit(X_train_c, y_train_c)
rfc_best = rfc_grid.best_estimator_
rfc_pred = rfc_best.predict(X_test_c)
rfc_cv = cross_val_score(rfc_best, X_train_c, y_train_c, cv=5, scoring='f1_weighted').mean()
print(f"Accuracy  = {accuracy_score(y_test_c, rfc_pred):.4f}")
print(f"F1 (weighted) = {f1_score(y_test_c, rfc_pred, average='weighted'):.4f}")
print(f"CV-F1     = {rfc_cv:.4f}")
print(f"Meilleurs params : {rfc_grid.best_params_}")

In [ ]:
# Matrices de confusion
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, cm_pred, name in zip(axes, [svm_pred, rfc_pred], ['SVM', 'RF Classifier']):
    cm = confusion_matrix(y_test_c, cm_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=le.classes_, yticklabels=le.classes_)
    ax.set_title(f'Matrice de Confusion – {name}', fontweight='bold')
    ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')
plt.tight_layout()
plt.show()
print("
=== Classification Report (Random Forest) ===")
print(classification_report(y_test_c, rfc_pred, target_names=le.classes_))

## 6. Sauvegarde des Modèles <a name="6"></a>

In [ ]:
os.makedirs('models', exist_ok=True)
joblib.dump(dt_best,  'models/dt_regressor.pkl')
joblib.dump(rf_best,  'models/rf_regressor.pkl')
joblib.dump(svm_best, 'models/svm_classifier.pkl')
joblib.dump(rfc_best, 'models/rfc_classifier.pkl')
joblib.dump(le,       'models/label_encoder.pkl')
print("✓ Tous les modèles sauvegardés avec joblib")
for f in os.listdir('models'):
    size = os.path.getsize(f'models/{f}') / 1024
    print(f"  {f:<35} {size:>8.1f} KB")

---
## Conclusion Générale

| Tâche | Modèle Retenu | Métrique Principale | Score |
|---|---|---|---|
| Régression | Random Forest Regressor | R² | 0.803 |
| Classification | Random Forest Classifier | F1-weighted | 0.443 |

**Régression :** Le Random Forest offre une précision remarquable avec un R²=0.803 et MAE≈14 800$, nettement supérieur au Decision Tree (R²=0.604).

**Classification :** Le déséquilibre prononcé des classes (1Fam ~60%) limite les performances. Des pistes d'amélioration incluent SMOTE, pondération des classes, ou l'acquisition de données supplémentaires pour les classes minoritaires.

---
*MUGENI KANZA CHRISTIAN — Master IA — Intelligence Artificielle*